In [2]:
#imports
import lightgbm as lgb
import sys
import numpy as np
import pandas as pd
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#read and create data frames
df1 = pd.read_csv("dataset/comments1.csv")

In [3]:
#get an idea of how the dataframe looks like
print(df1.head())
print(df1.shape)
print(df1.columns)

              kind  commentId  channelId  videoId  authorId  \
0  youtube#comment    1781382      14492    74288   2032536   
1  youtube#comment     289571      14727    79618   3043229   
2  youtube#comment     569077       3314    51826    917006   
3  youtube#comment    2957962       5008    58298   1853470   
4  youtube#comment     673093      21411     1265   2584166   

                                        textOriginal  parentCommentId  \
0  PLEASE LESBIAN FLAG I BEG YOU \n\nYou would ro...              NaN   
1   Apply mashed potato juice and mixed it with curd        3198066.0   
2                         69 missed calls from mars👽              NaN   
3                                               Baaa              NaN   
4    you look like raven from phenomena raven no cap              NaN   

   likeCount                publishedAt                  updatedAt  
0          0  2023-08-15 21:48:52+00:00  2023-08-15 21:48:52+00:00  
1          0  2023-10-02 13:08:22+00:00  202

In [4]:
#compare textOriginal, likeCount, publishedAt
trend1 = df1[["textOriginal", "likeCount", "publishedAt"]]
trend1.isnull().sum()

textOriginal    46
likeCount        0
publishedAt      0
dtype: int64

In [5]:
#clean up the dataframe
trend1 = trend1.dropna(subset=["textOriginal"])
trend1.head()

,textOriginal,likeCount,publishedAt
0,PLEASE LESBIAN FLAG I BEG YOU \n\nYou would ro...,0,2023-08-15 21:48:52+00:00
1,Apply mashed potato juice and mixed it with curd,0,2023-10-02 13:08:22+00:00
2,69 missed calls from mars👽,0,2024-05-31 12:03:12+00:00
3,Baaa,0,2024-02-13 15:48:37+00:00
4,you look like raven from phenomena raven no cap,0,2020-02-15 22:28:44+00:00


In [6]:
trend1['publishedAt'] = pd.to_datetime(trend1['publishedAt'], format='%Y-%m-%d %H:%M:%S%z', utc=True)
oldest = trend1['publishedAt'].min()
newest = trend1['publishedAt'].max()
oldest 
newest  

Timestamp('2025-07-20 15:09:26+0000', tz='UTC')

In [7]:
first_date = trend1["publishedAt"].min()
trend1["days_since_first"] = (trend1["publishedAt"] - first_date).dt.days.astype(float)

In [ ]:
filtered_likes_df1 = trend1[trend1["likeCount"] > 0]
average_like_count1 = filtered_likes_df1["likeCount"].mean()
successful_comments_1 = trend1[trend1["likeCount"] >= average_like_count1]

In [ ]:
#do what we did above
all_df = ["comments1.csv","comments2.csv", "comments3.csv", "comments4.csv", "comments5.csv"]
dfs = []  # list to hold each dataframe
for fname in all_df:
    df = pd.read_csv("dataset/" + fname)
    dfs.append(df)

# vertical concat (stack rows)
total_df = pd.concat(dfs, ignore_index=True)
trend = total_df[["textOriginal", "likeCount", "publishedAt"]]
trend = trend.dropna(subset=["textOriginal"])
trend['publishedAt'] = pd.to_datetime(trend['publishedAt'], format='%Y-%m-%d %H:%M:%S%z', utc=True)
first_date = trend["publishedAt"].min()
trend["days_since_first"] = (trend["publishedAt"] - first_date).dt.days.astype(float)

filtered_likes_df = trend[trend["likeCount"] > 0]
average_like_count = filtered_likes_df["likeCount"].mean()
successful_comments = trend[trend["likeCount"] >= average_like_count1]



NameError: name 'first_date' is not defined

In [9]:
#input the filtered dataframe that only has higher than average like counts for the parameter
def textOriginalAnalysis(df)-> dict:
    dfT = df["textOriginal"] #dfT short for dfText
    textArray = []
    stop_words = set(stopwords.words('english'))
    for i in dfT:  
        # ?:^ checks for if it is the start, can accept no whitespace if at the start
        words = re.findall(r'(?:^|\s)([1-9][0-9]*|[a-zA-Z]+)(?=\s|$)', i.lower(), flags=re.IGNORECASE)

        filtered_words = [word for word in words if word not in stop_words]

        textArray.extend(filtered_words)

    frequency_dict = Counter(textArray)
    topTenPercent = int(len(frequency_dict) / 10)   
    maxcount = frequency_dict.most_common(1)[0][1] #the count of the most common word of all
    
        # Function to convert a single comment into numeric value
    def comment_to_value(comment: str) -> float:
        words = re.findall(r'(?:^|\s)([1-9][0-9]*|[a-zA-Z]+)(?=\s|$)', 
                           str(comment).lower(), flags=re.IGNORECASE)
        filtered_words = [word for word in words if word not in stop_words]
        if not filtered_words:
            return 0.0
        # Average normalized frequency
        values = [frequency_dict[w] / maxcount for w in filtered_words if w in frequency_dict]
        return sum(values) / len(values) if values else 0.0

    # Add new column
    df = df.copy()
    df["commentValue"] = dfT.apply(comment_to_value)

    return df

def log1p_mapper(like : float) -> float:
    values = np.log1p(like)
    return values

analyzed_words = textOriginalAnalysis(successful_comments_1)
analyzed_words["likeCountLn"] = analyzed_words['likeCount'].apply(log1p_mapper)

    

In [10]:
analyzed_words

,textOriginal,likeCount,publishedAt,days_since_first,commentValue,likeCountLn
136,How do you achieve that slick back? 🧐,684,2025-01-28 09:03:05+00:00,1850.0,0.009693,6.529419
452,"“If it rains, I’m ruined” was so REALL😭😮😢",88,2024-07-13 05:19:05+00:00,1651.0,0.000000,4.488636
533,Can we please go back to Classic beauty?,75,2024-06-11 04:17:52+00:00,1619.0,0.084006,4.330733
573,POPULAR..YOURE GONNA BE POPULARR!,701,2024-11-25 15:03:24+00:00,1787.0,0.070275,6.553933
744,"Bro my skincare routine is soap, water and a t...",808,2023-04-29 16:40:35+00:00,1211.0,0.039849,6.695799
...,...,...,...,...,...,...
999631,gorgeous darling <3,245,2021-09-24 20:39:08+00:00,629.0,0.088853,5.505332
999799,Me who goes anywhere with just a face wash,304,2023-05-02 13:59:24+00:00,1213.0,0.088651,5.720312
999806,"I feel like U have more of a wave pattern, but...",2262,2024-11-22 03:04:23+00:00,1783.0,0.222132,7.724447
999821,Name the girls id you will get all information...,57,2023-08-03 20:24:14+00:00,1307.0,0.072429,4.060443


In [11]:
#machine learning imports
from lightgbm import LGBMRegressor   # or LGBMClassifier if you choose a binary label
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score

In [ ]:
# features and label
X = analyzed_words[["textOriginal", "days_since_first"]] #feature
y = analyzed_words["likeCountLn"]    # or df["commentValue"]

# text + numeric preprocessing
tfidf = TfidfVectorizer(max_features=4000, ngram_range=(1,2), min_df=5) #https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
#only transform the features for training, they become a matrix
pre = ColumnTransformer([
    ("txt", tfidf, "textOriginal"),
    ("num", "passthrough", ["days_since_first"]),
])

#model receives feature matrix and y_train to create model
model = LGBMRegressor(
    n_estimators=600, #num of trees
    learning_rate=0.05, #slower learning rate per tree but more stable
    subsample=0.9, #each tree takes 90 percent of the training dataset to avoid overfitting
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1
)

pipe = Pipeline([
    ("pre", pre),
    ("model", model),
])

# train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)

# evaluate
y_pred = pipe.predict(X_val)
print("RMSE:", root_mean_squared_error(y_val, y_pred))
print("R^2:", r2_score(y_val, y_pred))

# add predictions back
analyzed_words["pred"] = pipe.predict(X)
analyzed_words["pred_likeCount"] = np.expm1(analyzed_words["pred"])
analyzed_words[["textOriginal","likeCount","pred","pred_likeCount"]].head()

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.031657 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 32090
[LightGBM] [Info] Number of data points in the train set: 7502, number of used features: 1137
[LightGBM] [Info] Start training from score 5.399474
RMSE: 1.3679395527695297
R^2: -0.07605148341146761


,textOriginal,likeCount,pred,pred_likeCount
136,How do you achieve that slick back? 🧐,684,6.326676,558.294426
452,"“If it rains, I’m ruined” was so REALL😭😮😢",88,4.938490,138.559416
533,Can we please go back to Classic beauty?,75,4.921595,136.221347
573,POPULAR..YOURE GONNA BE POPULARR!,701,5.643875,281.555434
744,"Bro my skincare routine is soap, water and a t...",808,6.924077,1015.455352
